In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

seed = 123

# Data

In [2]:
# display number given a single flatten matrix

def show(matrix:np.ndarray):
    """Display a number given a single flatten matrix

    Args:
        matrix (np.ndarray): 1-D array of 784 elements representing a 28x28 image

    Returns:
        plt.imshow object: Display the image
    """
    return plt.imshow(matrix.reshape(28, 28), cmap='gray')

In [3]:
train_path = '../data/train.csv'
test_path = '../data/test.csv'
submission_path = '../data/sample_submission.csv'

In [7]:
class Data:
    """attributes:
            - Train (pd.DataFrame): original train data
            - Test (pd.DataFrame): original test data
            - Submission (pd.DataFrame): original submission data
            - train (pd.DataFrame): train data after splitting
            - cv1 (pd.DataFrame): cv1 data after splitting
            - cv2 (pd.DataFrame): cv2 data after splitting
            - *x_train (pd.DataFrame): x data for train
            - *y_train (pd.DataFrame): y data for train
            - *x_cv1 (pd.DataFrame): x data for cv1
            - *y_cv1 (pd.DataFrame): y data for cv1
            - *x_cv2 (pd.DataFrame): x data for cv2
            - *y_cv2 (pd.DataFrame): y data for cv2
    """
    def __init__(
        self,
        train: str,
        test: str,
        submission: str,
        cv1_size: float = 0.1,
        cv2_size: float = 0.1,
    ) -> None:
        """Data class to load the data and split it into train, test and submission dataframes

        Args:
            train (str): path for train.csv
            test (str): path for test.csv
            submission (str): path for sample_submission.csv
            cv1_size (float, optional): size of cv1 relative to train. Defaults to 0.1.
            cv2_size (float, optional): size of cv2 relative to train. Defaults to 0.1.
        """
        # import original data
        self.Train = pd.read_csv(train)
        self.Test = pd.read_csv(test)
        self.Submission = pd.read_csv(submission)
        
        # split data into train, cv1, cv2
        self.train, _test = train_test_split(
            self.Train, test_size=cv1_size + cv2_size, random_state=seed, stratify=self.Train["label"]
        )
        self.cv1, self.cv2 = train_test_split(
            _test, test_size=cv2_size / (cv1_size + cv2_size), random_state=seed, stratify=_test["label"]
        )
        del _test
        
        # split data into X, y
        self.x_train = self.train.drop(columns=["label"])
        self.y_train = self.train["label"]
        self.x_cv1 = self.cv1.drop(columns=["label"])
        self.y_cv1 = self.cv1["label"]
        self.x_cv2 = self.cv2.drop(columns=["label"])
        self.y_cv2 = self.cv2["label"]

In [ ]:
d = Data(
    train_path,
    test_path,
    submission_path,
    cv1_size=0.1,
    cv2_size=0.1,
)

> input shape (B,784)

# Model

In [ ]:
class Model(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        self.l1 = nn.Linear(784, 256)
        self.norm1 = nn.BatchNorm1d(256)
        self.a1 = nn.ReLU()
        self.l2 = nn.Linear(256, 128)
        self.norm2 = nn.BatchNorm1d(128)
        self.a2 = nn.ReLU()
        self.l3 = nn.Linear(128, 64)
        self.norm3 = nn.BatchNorm1d(64)
        self.a3 = nn.ReLU()
        self.l4 = nn.Linear(64, 10)
        
        self.sequential = nn.Sequential(
            self.l1,
            self.norm1,
            self.a1,
            self.l2,
            self.norm2,
            self.a2,
            self.l3,
            self.norm3,
            self.a3,
            self.l4,
        )
    def forward(self, x):
        return self.sequential(x)
    def loss(self, y_pred, y_true):
        return F.cross_entropy(y_pred, y_true)
    